In [ ]:
import numpy as np
import pandas as pd
import time
import threading
from collections import deque
from scipy.linalg import pinv
from dhanhq import dhanhq

In [ ]:
%pip install numpy pandas torch signatory scipy

     ---------------------------------------- 0.0/62.8 kB ? eta -:--:--
     ------------ ------------------------- 20.5/62.8 kB 640.0 kB/s eta 0:00:01
     ---------------------------------------- 62.8/62.8 kB 1.1 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [26 lines of output]
      Traceback (most recent call last):
        File "<string>", line 21, in <module>
      ModuleNotFoundError: No module named 'torch'
      
      During handling of the above exception, another exception occurred:
      
      Traceback (most recent call last):
        File "c:\GIT\.venv\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 353, in <module>
          main()
        File "c:\GIT\.venv\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 335, in main
          json_out['return_val'] = hook(**hook_input['kwargs'])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "c:\GIT\.venv\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 118, in get_requires_for_build_wheel
          return hook(config_settings)
          

: 

In [ ]:


#############################################
# CONFIGURATION
#############################################

CLIENT_ID = "YOUR_CLIENT_ID"
ACCESS_TOKEN = "YOUR_ACCESS_TOKEN"

SYMBOL = "NIFTY"
SECURITY_ID = "13"  # Example NIFTY index
EXCHANGE = "NSE_INDEX"

WINDOW_SIZE = 40
THRESHOLD_PERCENTILE = 0.98

TRADE_QTY = 50
MAX_POSITION = 200

#############################################
# DHAN CONNECTION
#############################################

dhan = dhanhq(CLIENT_ID, ACCESS_TOKEN)

#############################################
# DATA STORAGE
#############################################

price_buffer = deque(maxlen=WINDOW_SIZE)
feature_history = []
positions = 0

#############################################
# VARIANCE NORM
#############################################

def variance_norm(x, X):

    X = np.array(X)

    mu = X.mean(axis=0)

    cov = np.cov(X.T)

    inv_cov = pinv(cov)

    d = (x - mu).T @ inv_cov @ (x - mu)

    return d


#############################################
# FEATURE BUILDER
#############################################

def build_feature(segment):

    segment = np.array(segment)

    ret = np.diff(segment)

    mean = np.mean(ret)
    std = np.std(ret)
    max_move = np.max(np.abs(ret))

    return np.array([mean, std, max_move])


#############################################
# JUMP DETECTOR
#############################################

class JumpDetector:

    def __init__(self):

        self.threshold = None

    def update(self, price):

        price_buffer.append(price)

        if len(price_buffer) < WINDOW_SIZE:
            return None

        segment = list(price_buffer)

        feature = build_feature(segment)

        if len(feature_history) < 50:
            feature_history.append(feature)
            return None

        score = variance_norm(feature, feature_history)

        feature_history.append(feature)

        scores = []

        for f in feature_history:
            scores.append(variance_norm(f, feature_history))

        self.threshold = np.quantile(scores, THRESHOLD_PERCENTILE)

        jump = score > self.threshold

        return jump, score


detector = JumpDetector()

#############################################
# ORDER MANAGEMENT
#############################################

def place_order(side):

    global positions

    if side == "BUY" and positions >= MAX_POSITION:
        return

    if side == "SELL" and positions <= -MAX_POSITION:
        return

    order = dhan.place_order(
        security_id=SECURITY_ID,
        exchange_segment=EXCHANGE,
        transaction_type=side,
        quantity=TRADE_QTY,
        order_type="MARKET",
        product_type="INTRADAY",
        price=0
    )

    print("ORDER:", order)

    if side == "BUY":
        positions += TRADE_QTY
    else:
        positions -= TRADE_QTY


#############################################
# STRATEGY LOGIC
#############################################

def strategy(price):

    result = detector.update(price)

    if result is None:
        return

    jump, score = result

    if jump:

        print("JUMP DETECTED:", price, score)

        if score > detector.threshold * 1.2:

            place_order("BUY")

        elif score < detector.threshold * 0.8:

            place_order("SELL")


#############################################
# MARKET DATA STREAM
#############################################

def market_data_stream():

    print("Starting market stream")

    while True:

        try:

            data = dhan.get_ltp_data(
                exchange_segment=EXCHANGE,
                security_id=SECURITY_ID
            )

            price = float(data["data"]["last_price"])

            strategy(price)

            time.sleep(1)

        except Exception as e:

            print("Stream error:", e)
            time.sleep(2)


#############################################
# RISK MONITOR
#############################################

def risk_monitor():

    global positions

    while True:

        if abs(positions) > MAX_POSITION:

            print("Risk limit reached")

        time.sleep(5)


#############################################
# LOGGING
#############################################

def log_system():

    while True:

        print("Positions:", positions)
        print("Buffer size:", len(price_buffer))
        print("History size:", len(feature_history))

        time.sleep(10)


#############################################
# THREAD RUNNER
#############################################

def main():

    t1 = threading.Thread(target=market_data_stream)
    t2 = threading.Thread(target=risk_monitor)
    t3 = threading.Thread(target=log_system)

    t1.start()
    t2.start()
    t3.start()

    t1.join()
    t2.join()
    t3.join()


#############################################
# START
#############################################

if __name__ == "__main__":

    main()